# PinchBench - Per-Task Deep Dive

Interaktives Notebook fuer Detail-Analyse einzelner PinchBench-Tasks.

**Workflow:**
1. Tasks browsen / filtern.
2. Ein Task auswaehlen.
3. Agent bauen (frisch pro Run), Workspace provisionieren.
4. Mission laufen lassen - jedes Event wird capture'd.
5. Text-Berichte ansehen (tool stats, context evolution, ALLE tool
   results inkl. Output, workspace-diff, final answer).
6. Grafiken: tool frequencies, context-growth, role distribution,
   tool-timeline, per-criterion grading bar chart.
7. Grading: automated check + optional LLM-judge.
8. Failure-Analyse mit allen Diagnostik-Signalen.

Stuetzt sich auf ``notebooks/analysis_lib.py``. Mission-Setup und Grading
spiegeln ``evals/pinchbench/solver.py`` und ``evals/pinchbench/scorer.py``,
aber alles laeuft hier sichtbar im Notebook (nicht im Inspect-AI-Wrapper).


## 1. Setup


In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks'), str(PYTF_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

# Inline matplotlib + nice defaults.
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

from evals.bridge.azure_config import setup_azure_env, DEFAULT_MODEL
setup_azure_env()

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib

print('analysis_lib OK')
print(f'  pytaskforce  : {PYTF_ROOT}')
print(f'  default model: {DEFAULT_MODEL}')


## 2. Factory, Executor, Pinchbench-Profil registrieren


In [ ]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor
from taskforce.application.bootstrap_config_dirs import bootstrap_config_dirs

registered = bootstrap_config_dirs(force=True)
print(f'Registered config dirs ({len(registered)}):')
for d in registered:
    print(f'  - {d}')

factory  = AgentFactory()
executor = AgentExecutor(factory=factory)
alib.disable_post_mission_learning(executor)

async def build_agent():
    return await factory.create_agent(profile='pinchbench')

_smoke = await build_agent()
print()
print(f'Pinchbench agent smoke-built OK')
print(f'  Tools                 : {sorted(_smoke.tools.keys())}')
print(f'  Planning strategy     : {_smoke.planning_strategy.name}')
print(f'  Max steps             : {_smoke.max_steps}')
print(f'  Max parallel tools    : {_smoke.max_parallel_tools}')
print(f'  summary_threshold     : {_smoke.message_history_manager._summary_threshold}')
# Threshold lives on the agent itself (set by factory from YAML).
print(f'  tool_result_store thr : {_smoke._tool_result_store_threshold}')
await _smoke.close()

## 3. PinchBench-Tasks laden


In [ ]:
from evals.pinchbench.loader import load_tasks
import pandas as pd

tasks = load_tasks(suite='all')

df = pd.DataFrame([{
    'id': t.id,
    'category': t.category,
    'grading_type': t.grading_type,
    'timeout_s': t.timeout_seconds,
    'has_grade_fn': bool(t.grade_function),
    'has_rubric': bool(t.llm_judge_rubric),
    'workspace_files_n': len(t.workspace_files),
    'multi_session': bool(t.multi_session_prompts),
    'prompt_len': len(t.prompt),
} for t in tasks])

print(f'Total tasks: {len(tasks)}')
print()
print('Per-category overview:')
print(df.groupby('category').agg(
    n=('id', 'count'),
    automated=('grading_type', lambda s: (s == 'automated').sum()),
    hybrid=('grading_type', lambda s: (s == 'hybrid').sum()),
    llm_judge=('grading_type', lambda s: (s == 'llm_judge').sum()),
).sort_values('n', ascending=False).to_string())


In [ ]:
# Visual overview: tasks per category, grouped by grading_type.
cat_grade = df.groupby(['category', 'grading_type']).size().unstack(fill_value=0)
cat_grade = cat_grade.sort_values(cat_grade.columns.tolist(), ascending=False)

fig, ax = plt.subplots(figsize=(11, max(4, 0.4 * len(cat_grade))))
cat_grade.plot(kind='barh', stacked=True, ax=ax,
               color={'automated': '#3a7d44', 'hybrid': '#e07b39', 'llm_judge': '#7a52a8'})
ax.invert_yaxis()
ax.set_xlabel('Number of tasks')
ax.set_title(f'PinchBench task distribution (n={len(tasks)})')
ax.legend(title='Grading type', loc='lower right')
for cat, vals in cat_grade.iterrows():
    total = vals.sum()
    ax.text(total + 0.3, list(cat_grade.index).index(cat), str(total),
            va='center', fontsize=9, color='dimgray')
plt.tight_layout()
plt.show()


In [ ]:
FILTER_CATEGORY = None         # e.g. 'meeting_analysis' or None
FILTER_GRADING  = None         # e.g. 'automated' or None
FILTER_SUBSTR   = None         # e.g. 'csv_stock' or None

filtered = tasks
if FILTER_CATEGORY:
    filtered = [t for t in filtered if t.category == FILTER_CATEGORY]
if FILTER_GRADING:
    filtered = [t for t in filtered if t.grading_type == FILTER_GRADING]
if FILTER_SUBSTR:
    filtered = [t for t in filtered if FILTER_SUBSTR in t.id]

print(f'Matching tasks: {len(filtered)}')
print()
for t in filtered[:80]:
    print(f'  {t.id:42s}  [{t.category:18s}]  {t.grading_type:10s}  ws={len(t.workspace_files):2d}  prompt={len(t.prompt):4d}c')
if len(filtered) > 80:
    print(f'  ... +{len(filtered)-80} more (raise the slice if you want)')


## 4. Task auswaehlen + Detail-View


In [ ]:
TARGET_TASK_ID = 'task_oss_alternative_research'   # was 0.00 — re-run after store_dir absolute-path fix

target = next((t for t in tasks if t.id == TARGET_TASK_ID), None)
if target is None:
    raise SystemExit(f'Task {TARGET_TASK_ID!r} not found. Browse above for valid IDs.')

print(f'== {target.id} ==')
print(f'Name        : {target.name}')
print(f'Category    : {target.category}')
print(f'Grading     : {target.grading_type}')
print(f'Timeout     : {target.timeout_seconds}s')
print(f'Workspace   : {target.workspace_files}')
print(f'Multi-session: {bool(target.multi_session_prompts)}')
print()
print('--- PROMPT ---')
print(target.prompt)

In [ ]:
if target.grade_function:
    print('--- AUTOMATED GRADE FUNCTION ---')
    print(target.grade_function[:3000])
    if len(target.grade_function) > 3000:
        print(f'... [+{len(target.grade_function)-3000} chars truncated]')
else:
    print('(no automated grade function for this task)')

print()
if target.llm_judge_rubric:
    print('--- LLM JUDGE RUBRIC ---')
    print(target.llm_judge_rubric[:2500])
    if len(target.llm_judge_rubric) > 2500:
        print(f'... [+{len(target.llm_judge_rubric)-2500} chars truncated]')
else:
    print('(no LLM-judge rubric)')

if target.expected_behavior:
    print()
    print('--- EXPECTED BEHAVIOR ---')
    print(target.expected_behavior[:1500])

if target.grading_criteria:
    print()
    print('--- GRADING CRITERIA ---')
    print(target.grading_criteria[:1500])


## 5. Workspace provisionieren + Prompt augmentieren


In [ ]:
from evals.pinchbench.solver import _provision_workspace, _augment_prompt

workspace = _provision_workspace(list(target.workspace_files))
augmented_prompt = _augment_prompt(target.prompt, workspace)

print(f'Workspace : {workspace}')
print()
print('Workspace contents:')
for entry in sorted(workspace.rglob('*')):
    if entry.is_file():
        size_kb = entry.stat().st_size / 1024
        rel = entry.relative_to(workspace)
        print(f'  {rel}  ({size_kb:.1f} KB)')

print()
print(f'Augmented prompt ({len(augmented_prompt)} chars, last 250):')
print(augmented_prompt[-250:])


## 6. Mission ausfuehren


In [ ]:
agent = await build_agent()
SNAP_DIRS = ('',)

rec = await alib.run(
    executor, agent, augmented_prompt,
    project_root=workspace,
    snapshot_subdirs=SNAP_DIRS,
    initial_system_prompt_chars=sum(
        len(str(m.get('content',''))) for m in agent.context.messages if m.get('role') == 'system'
    ),
    max_print_events=60,
)
print()
print('=' * 60)
alib.print_summary(rec)


## 7. Text-Berichte


In [ ]:
alib.print_tool_stats(rec)


In [ ]:
alib.print_tool_results(rec, head=None)


In [ ]:
alib.print_context_evolution(rec)


In [ ]:
alib.print_sanity_check(rec)


In [ ]:
alib.print_context_view(agent, rec)


## 8. Grafische Auswertung des Runs

Sechs Charts pro Run:

- **Tool frequencies** : welche Tools wie oft.
- **Context growth**   : kumulative Tool-Output-Chars ueber die Zeit
  (Proxy fuer Context-Last und Compression-Druck).
- **Role distribution**: Messages pro Rolle (system / user / assistant
  / tool) + Chars pro Rolle.
- **Tool timeline**    : wann jeder Tool-Call gefeuert hat, mit
  success/failure color-coded.
- **Token utilization**: wie viel Token-Budget am Ende belegt war.


In [ ]:
fig = alib.plot_tool_frequencies(rec, title=f'Tool-Aufrufe: {target.id}')
plt.show()


In [ ]:
fig = alib.plot_context_growth(rec, title=f'Context-Wachstum: {target.id}')
plt.show()


In [ ]:
fig = alib.plot_role_distribution(agent, title=f'Context-Verteilung: {target.id}')
plt.show()


In [ ]:
# Tool-call timeline. Each tool call is a marker at (t, tool); color
# encodes success (green) vs failure (red). Useful for spotting long
# gaps (LLM thinking), bursts (parallel-tool), and where stalls happen.
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

calls = [e for e in rec.events if e['event'] == 'tool_call']
# Pair each tool_call with the next tool_result for success info.
results_by_t: dict[float, bool] = {}
for ev in rec.events:
    if ev['event'] == 'tool_result':
        results_by_t[ev['t']] = bool(ev.get('tool_success'))

# Map success status forward to the call by event order.
status_per_call: list[bool] = []
result_iter = iter([e for e in rec.events if e['event'] == 'tool_result'])
for _ in calls:
    try:
        status_per_call.append(bool(next(result_iter).get('tool_success')))
    except StopIteration:
        status_per_call.append(False)

unique_tools = sorted({c['tool'] for c in calls if c.get('tool')})
ypos = {t: i for i, t in enumerate(unique_tools)}

fig, ax = plt.subplots(figsize=(11, max(3, 0.45 * len(unique_tools))))
for c, ok in zip(calls, status_per_call):
    tool = c.get('tool') or '?'
    if tool not in ypos:
        continue
    ax.scatter(c['t'], ypos[tool],
               c='#3a7d44' if ok else '#c0392b',
               s=60, alpha=0.85, edgecolors='black', linewidths=0.5)

ax.set_yticks(list(ypos.values()))
ax.set_yticklabels(list(ypos.keys()))
ax.set_xlabel('Time (s)')
ax.set_title(f'Tool-Timeline: {target.id}  (gruen=success, rot=failure)')
ax.invert_yaxis()
ax.grid(alpha=0.3)

# Annotate stream restarts (content-filter recovery) as vertical lines.
for sr in rec.stream_restarts:
    ax.axvline(sr['t'], color='purple', linestyle='--', alpha=0.5,
               label='stream_restart')
if rec.stream_restarts:
    ax.legend(handles=[Line2D([0], [0], color='purple', linestyle='--',
                              label='stream_restart')], loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# Token-budget snapshot. Stacked bar shows how the budget is split
# across system_prompt / messages / memory / skills / tools.
snap = agent.context.snapshot(include_content=True)

categories = [
    ('system_prompt', snap.system_prompt, '#34495e'),
    ('messages',     snap.messages,      '#3498db'),
    ('memory',       snap.memory,        '#9b59b6'),
    ('skills',       snap.skills,        '#e67e22'),
    ('tools',        snap.tools,         '#16a085'),
]
totals = {label: sum(i.tokens for i in items) for label, items, _ in categories}
labels = list(totals.keys())
values = list(totals.values())
colors = [c for _, _, c in categories]

fig, ax = plt.subplots(figsize=(9, 4))
left = 0
for label, val, color in zip(labels, values, colors):
    ax.barh([0], [val], left=left, color=color, label=f'{label} ({val})')
    if val > snap.max_tokens * 0.02:
        ax.text(left + val / 2, 0, str(val), ha='center', va='center',
                color='white', fontsize=10, fontweight='bold')
    left += val
ax.axvline(snap.max_tokens, color='red', linestyle='--', alpha=0.7,
           label=f'max_tokens ({snap.max_tokens})')
ax.set_yticks([])
ax.set_xlabel('Tokens')
ax.set_title(
    f'Token-Budget: {snap.total_tokens}/{snap.max_tokens} '
    f'({snap.utilization_percent:.1f}% utilization)'
)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()


## 9. Workspace-Inspektion


In [ ]:
diff = alib.file_diff_report(rec)
print(f'New files     : {len(diff["new"])}')
for f in diff['new']:
    full = workspace / f
    size = full.stat().st_size if full.exists() else 0
    print(f'  + {f}  ({size} bytes)')
print(f'Modified files: {len(diff["modified"])}')
for f in diff['modified']:
    full = workspace / f
    size = full.stat().st_size if full.exists() else 0
    print(f'  ~ {f}  ({size} bytes)')
print(f'Deleted       : {len(diff["deleted"])}')
print(f'Unchanged     : {len(diff["unchanged"])} (fixtures incl.)')


In [ ]:
MAX_CHARS = 4000
for f in diff['new'] + diff['modified']:
    full = workspace / f
    print('=' * 60)
    print(f'FILE: {f}')
    print('=' * 60)
    try:
        text = full.read_text(encoding='utf-8', errors='replace')
    except Exception as exc:
        print(f'(could not read: {exc})')
        continue
    if len(text) > MAX_CHARS:
        print(text[:MAX_CHARS])
        print(f'... [+{len(text)-MAX_CHARS} chars truncated]')
    else:
        print(text)
    print()


In [ ]:
print('--- AGENT FINAL ANSWER ---')
print(rec.final_answer or '(empty)')


## 10. Grading


In [ ]:
from evals.pinchbench.transcript import build_transcript
from types import SimpleNamespace

# build_transcript expects ProgressUpdate-like objects with attribute
# access (.event_type, .details, .message). alib.run stores dicts, so
# wrap each event in a shim before passing it in.
def _to_evt(d):
    return SimpleNamespace(
        event_type=d.get('event', ''),
        details=d.get('details') or {
            # Re-hydrate the relevant fields back into details — build_transcript
            # reads tool name/args from details.tool / details.args and
            # tool output from details.output.
            'tool':   d.get('tool'),
            'args':   d.get('tool_args'),
            'output': d.get('output'),
        },
        message=d.get('message', '') or '',
    )

evt_shim = [_to_evt(d) for d in rec.events]
transcript = build_transcript(evt_shim, target.prompt)
print(f'Transcript entries: {len(transcript)}')

from collections import Counter
breakdown = Counter()
for e in transcript:
    msg = e.get('message') or {}
    role = msg.get('role', '?')
    for c in (msg.get('content') or []):
        breakdown[(role, c.get('type'))] += 1
for (role, ctype), n in sorted(breakdown.items(), key=lambda x: -x[1]):
    print(f'  {role:10s}  {ctype:14s}  {n:4d}')

In [ ]:
USE_LLM_JUDGE = True   # enabled for pure-judge tasks like competitive_research

from evals.pinchbench.grading import run_automated_check, aggregate_scores, run_llm_judge

components = {}
details = {'task_id': target.id, 'grading_type': target.grading_type}

if target.grading_type in ('automated', 'hybrid') and target.grade_function:
    print(f'Running automated grader (timeout={min(30, target.timeout_seconds)}s)...')
    auto = run_automated_check(
        target.grade_function, transcript, workspace,
        timeout_seconds=min(30, target.timeout_seconds),
    )
    if auto.get('ok'):
        scores = auto.get('scores') or {}
        components['automated'] = aggregate_scores(scores)
        details['automated_scores'] = scores
        print(f'  ok, mean={components["automated"]:.3f}')
        for k, v in scores.items():
            marker = '[ok]' if v >= 0.99 else ('[..]' if v >= 0.5 else '[!!]')
            print(f'    {marker} {k:30s}  {v:.2f}')
    else:
        details['automated_error'] = auto.get('error', 'unknown')
        print(f'  FAILED: {auto.get("error")[:300]}')
        if 'traceback' in auto:
            print(auto['traceback'][:1500])
else:
    print('(no automated grader for this task)')

if USE_LLM_JUDGE and target.grading_type in ('llm_judge', 'hybrid'):
    print()
    print('Running LLM judge...')
    judge = await run_llm_judge(
        prompt=target.prompt,
        transcript=transcript,
        rubric=target.llm_judge_rubric,
        expected_behavior=target.expected_behavior,
        grading_criteria=target.grading_criteria,
    )
    if judge.get('ok'):
        components['judge'] = float(judge['score'])
        details['judge_reasoning'] = judge.get('reasoning', '')
        print(f'  ok, score={components["judge"]:.3f}')
        print(f'  reasoning: {judge.get("reasoning", "")}')
    else:
        details['judge_error'] = judge.get('error', '')
        print(f'  FAILED: {judge.get("error")[:300]}')

print()
print('=' * 60)
if components:
    final = sum(components.values()) / len(components)
    print(f'FINAL SCORE: {final*100:.1f}%   components={components}')
else:
    final = 0.0
    print('FINAL SCORE: 0.0%   (no usable grader)')

In [ ]:
# Per-criterion bar chart. Green >= 1.0, yellow >= 0.5, red below.
auto_scores = details.get('automated_scores') or {}
if not auto_scores:
    print('(no automated criteria to plot - either pure llm_judge or grader failed)')
else:
    crit, vals = zip(*auto_scores.items())
    colors = ['#27ae60' if v >= 0.99 else ('#f39c12' if v >= 0.5 else '#c0392b') for v in vals]
    fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(crit))))
    ax.barh(list(crit), list(vals), color=colors, edgecolor='black', linewidth=0.5)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Score (0.0 - 1.0)')
    ax.set_title(f'Per-Kriterium Scores: {target.id}')
    for i, v in enumerate(vals):
        ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9)
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.4)
    ax.axvline(1.0, color='gray', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
# Component breakdown (automated vs judge) - only meaningful for hybrid.
if len(components) >= 2:
    labels = list(components.keys())
    vals = [components[k] for k in labels]
    final_val = sum(vals) / len(vals) if vals else 0.0

    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar(labels + ['FINAL'], vals + [final_val],
                  color=['#3498db', '#9b59b6', '#2c3e50'])
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score (0.0 - 1.0)')
    ax.set_title(f'Component breakdown: {target.id}')
    for b, v in zip(bars, vals + [final_val]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02,
                f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
elif components:
    print(f'(only one component: {list(components.keys())[0]}={list(components.values())[0]:.3f} - skip chart)')
else:
    print('(no components to plot)')


## 11. Failure-Analyse


In [ ]:
auto_scores = details.get('automated_scores') or {}
if auto_scores:
    missing = [(k, v) for k, v in auto_scores.items() if v < 1.0]
    if missing:
        print('FAILED / PARTIAL criteria:')
        for k, v in missing:
            print(f'  {k:35s} = {v:.2f}')
    else:
        print('All automated criteria passed.')

writes_via_tool = rec.tool_calls.get('file_write', 0) + rec.tool_calls.get('edit', 0)
sub_agent_calls = rec.tool_calls.get('call_agents_parallel', 0)
print()
print(f'tool_calls          : {dict(rec.tool_calls)}')
print(f'file_write+edit     : {writes_via_tool}')
print(f'call_agents_parallel: {sub_agent_calls}')
print(f'event_count         : {len(rec.events)}')
print(f'duration            : {rec.duration:.1f}s')
print(f'llm_calls           : {rec.llm_calls}')
print(f'stream_restarts     : {len(rec.stream_restarts)}')
if rec.stream_restarts:
    for sr in rec.stream_restarts:
        print(f'  - stage={sr["stage"]} reason={sr["reason"]}')

if not diff['new'] and not diff['modified']:
    print()
    print('No new or modified files in workspace - deliverable was never written.')
elif diff['new']:
    print()
    print('Files agent wrote:')
    for f in diff['new']:
        full = workspace / f
        size = full.stat().st_size if full.exists() else 0
        print(f'  {f}  ({size} bytes)')


## 12. Cleanup


In [ ]:
import shutil
await agent.close()
if workspace.name.startswith('pinchbench_ws_') and workspace.exists():
    shutil.rmtree(workspace, ignore_errors=True)
    print(f'Cleaned up workspace: {workspace}')
print('Done. Pick a new TARGET_TASK_ID and rerun Section 4+.')


## 13. Optional: Kategorie-Mini-Sweep

Wenn du ein Pattern verstehst, lass alle Tasks einer Kategorie nacheinander
laufen und sieh, ob das Verhalten konsistent ist. Achtung: viel LLM-Time.
Ein einzelner Lauf dauert 30s-3min; eine Kategorie mit n=28 eher 30-90 min.


In [ ]:
CATEGORY_TO_SWEEP = None   # set to e.g. 'log_analysis' to actually run; None = noop

sweep_results = []
if CATEGORY_TO_SWEEP:
    sweep_tasks = [t for t in tasks if t.category == CATEGORY_TO_SWEEP]
    print(f'Sweep {CATEGORY_TO_SWEEP}: {len(sweep_tasks)} tasks')
    for st in sweep_tasks:
        sweep_ws = _provision_workspace(list(st.workspace_files))
        sweep_prompt = _augment_prompt(st.prompt, sweep_ws)
        sweep_agent = await build_agent()
        try:
            sweep_rec = await alib.run(
                executor, sweep_agent, sweep_prompt,
                project_root=sweep_ws, snapshot_subdirs=('',),
                max_print_events=0, silent=True,
            )
            sweep_transcript = build_transcript(sweep_rec.events, st.prompt)
            sweep_score = None
            if st.grade_function:
                a = run_automated_check(st.grade_function, sweep_transcript, sweep_ws,
                                        timeout_seconds=min(30, st.timeout_seconds))
                if a.get('ok'):
                    sweep_score = aggregate_scores(a.get('scores') or {})
            sweep_results.append({
                'id': st.id, 'score': sweep_score,
                'events': len(sweep_rec.events), 'tools': sum(sweep_rec.tool_calls.values()),
                'writes': sweep_rec.tool_calls.get('file_write', 0) + sweep_rec.tool_calls.get('edit', 0),
                'duration': sweep_rec.duration,
            })
            await sweep_agent.close()
        finally:
            import shutil as _sh
            if sweep_ws.name.startswith('pinchbench_ws_') and sweep_ws.exists():
                _sh.rmtree(sweep_ws, ignore_errors=True)
    import pandas as pd
    sweep_df = pd.DataFrame(sweep_results).sort_values('score', na_position='last')
    print(sweep_df.to_string(index=False))
else:
    print('(noop - set CATEGORY_TO_SWEEP to run)')


In [ ]:
# Horizontal bar of per-task scores in the swept category.
if not sweep_results:
    print('(no sweep results to plot)')
else:
    rows = sorted(sweep_results, key=lambda r: (r['score'] is None, r['score']))
    ids = [r['id'].removeprefix('task_') for r in rows]
    vals = [r['score'] if r['score'] is not None else 0 for r in rows]
    colors = []
    for r in rows:
        if r['score'] is None:
            colors.append('#7f8c8d')
        elif r['score'] >= 0.75:
            colors.append('#27ae60')
        elif r['score'] >= 0.25:
            colors.append('#f39c12')
        else:
            colors.append('#c0392b')

    fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(rows))))
    ax.barh(ids, vals, color=colors, edgecolor='black', linewidth=0.4)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Score (0.0 - 1.0)')
    ax.set_title(f'Sweep {CATEGORY_TO_SWEEP}: per-task scores')
    for i, (v, r) in enumerate(zip(vals, rows)):
        label = 'n/a' if r['score'] is None else f'{v*100:.0f}%'
        ax.text(v + 0.01, i, label, va='center', fontsize=9)
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
